# Rennes — historical cycling LTS and 2022 optimization network

Same Grenoble classification and contraction rules. **2017–2021:** cycling LTS 1–4. **2022:** bike LTS 1–4 plus missing car LTS 5, simplified **together**. Metrics use each already simplified graph and ignore LTS 5 edges. Plotly renders directly in Jupyter, with no Matplotlib plots. Each year is cached to a pickle.

Keep the `.py` modules provided alongside this notebook. Edit only the JSON file path in the first cell. You need an Internet connection to populate missing historical graph caches.

## 1. Polygon and city-specific paths

In [ ]:
from pathlib import Path
import pickle
import geojson
from shapely.geometry import shape
import networkx as nx
from IPython.display import display

CITY = "Rennes"
# Replace this with the actual location of your GeoJSON on your computer.
result_file = Path(r"C:\Users\Administrator\Documents\PhD Backup\PhD Backup\limites_administrative\Administrative_limites_villes\communes-20220101.json")

with result_file.open("r", encoding="utf-8") as f:
    gj = geojson.load(f)
feature = next(f for f in gj["features"] if f["properties"]["nom"] == CITY)
poly = shape(feature["geometry"])
if not poly.is_valid:
    poly = poly.buffer(0)
if poly.is_empty:
    raise ValueError("The selected city polygon is empty")

ROOT = Path("lts_output") / CITY.lower() / "historical"
CACHE = Path("cache_historical_osm") / CITY.lower()
ROOT.mkdir(parents=True, exist_ok=True)
print("City:", CITY, "Polygon:", poly.geom_type, "Valid:", poly.is_valid, "Bounds:", poly.bounds)
print("Output:", ROOT, "Cache:", CACHE)


## 2. Build and cache the six simplified networks

This step uses your original historical builder and LTS classifier. The original Matplotlib plotting function is disabled during network construction. An existing yearly pickle skips historical downloads; a 2022 combined pickle also skips the car download.

In [ ]:
from grenoble_workflow_fixed import get_bike_year, add_2022_car_layer, save_pickle, load_pickle
from grenoble_metrics_repair import prepare_metrics_source
from grenoble_simplification_exact import simplify_lts_graph, check_graph, plot_topology
from grenoble_metrics_exponential import annual_metrics, exponential_weights, save_txt

# False: use saved yearly results whenever possible. True: force a new build.
REBUILD = False
TIMEOUT_SECONDS = 600

def graph_for_year(year, *, rebuild=False):
    folder = ROOT / str(year)
    folder.mkdir(parents=True, exist_ok=True)
    saved = folder / (f"{CITY.lower()}_lts5_{year}_SIMPLE_optimization.pkl" if year == 2022
                      else f"{CITY.lower()}_lts4_{year}_SIMPLE_metrics.pkl")
    if saved.exists() and not rebuild:
        print(f"[{year}] Loaded simplified graph:", saved)
        return load_pickle(saved)

    bike_path = folder / f"{CITY.lower()}_bike_{year}_prepared.pkl"
    if bike_path.exists() and not rebuild:
        bike = load_pickle(bike_path)
    else:
        # Reuses the original working builder and its classifier; only prevents
        # Matplotlib from being invoked when the historical graph is downloaded.
        from grenoble_lts_historical import build_historical_lts_graph
        import grenoble_lts_graph as original
        old_plot = original.plot_lts_graph
        original.plot_lts_graph = lambda *args, **kwargs: None
        try:
            full, main, _, _ = build_historical_lts_graph(
                poly=poly, snapshot_date=f"{year}-12-31T23:59:59Z",
                output_dir=folder, requests_timeout=TIMEOUT_SECONDS,
                cache_root=CACHE,
            )
        finally:
            original.plot_lts_graph = old_plot
        bike = prepare_metrics_source(main)
        bike.graph['year'] = year
        save_pickle(bike, bike_path)
        print(f"[{year}] Saved bicycle checkpoint:", bike_path)

    if year == 2022:
        combined_path = folder / f"{CITY.lower()}_lts5_2022_pre_simplification.pkl"
        if combined_path.exists() and not rebuild:
            source = load_pickle(combined_path)
        else:
            source = add_2022_car_layer(
                bike, poly, timeout=TIMEOUT_SECONDS, cache_root=CACHE
            )
            save_pickle(source, combined_path)
            print("Saved pre-simplification combined graph:", combined_path)
    else:
        source = bike

    H = simplify_lts_graph(source)
    H.graph['year'] = year
    H.graph['city'] = CITY
    assert type(H) is nx.Graph and nx.is_connected(H)
    check = check_graph(H)
    assert check['remaining_contractible_degree2_nodes'] == 0
    if year < 2022:
        assert all(d['key'] <= 4 for _, _, d in H.edges(data=True))
    else:
        assert any(d['key'] == 5 for _, _, d in H.edges(data=True)), "No 2022 LTS5 roads found"
    save_pickle(H, saved)
    print(f"[{year}] Saved simplified graph:", saved)
    return H

# The first run may take time because historical OSM data must be downloaded.
# Successful years are saved immediately, so rerunning resumes from disk.
metric_graphs = {}
failed_years = {}
for year in range(2017, 2023):
    try:
        metric_graphs[year] = graph_for_year(year, rebuild=REBUILD)
    except Exception as exc:
        failed_years[year] = repr(exc)
        print(f"[{year}] FAILED: {exc}")

print("Completed:", sorted(metric_graphs))
print("Failed:", failed_years)


## 3. Annual normalized length and exponential-weighted LCC

Use the **already-simplified** graphs above. The 2022 source is the combined 1–5 simplified graph, but metrics count only LTS 1–4. The reference length is the maximum of the six annual cycling lengths by default. For comparable analyses across cities, specify a consistent reference policy.

In [ ]:
if len(metric_graphs) != 6:
    raise RuntimeError(f"Need all six years for the normalized time-series metrics; missing {sorted(set(range(2017,2023))-set(metric_graphs))}")

KAPPA = 1.0
REFERENCE_LENGTH_KM = None
metrics = annual_metrics(metric_graphs, kappa=KAPPA, reference_length_km=REFERENCE_LENGTH_KM)
columns = ["year", "cycling_length_km", "normalized_network_length",
           "lcc_lts1_km", "lcc_lts2_km", "lcc_lts3_km", "lcc_lts4_km",
           "weighted_lcc_normalized", "lts5_edges", "lts5_mandatory_edges"]
display(metrics[columns].round(5))

# Generic TXT writer: use the existing calculation's text formatter and
# replace only the city title so the file is correctly labeled Rennes.
txt_path = ROOT / f"{CITY.lower()}_2017_2022_SIMPLIFIED_exponential_metrics.txt"
csv_path = ROOT / f"{CITY.lower()}_2017_2022_SIMPLIFIED_exponential_metrics.csv"
save_txt(metrics, txt_path)
content = txt_path.read_text(encoding="utf-8").replace("Grenoble infrastructure metrics", f"{CITY} infrastructure metrics")
txt_path.write_text(content, encoding="utf-8")
metrics.to_csv(csv_path, index=False)
print("Saved:", txt_path, csv_path)
print("Weights:", exponential_weights(KAPPA))


## 4. Show the six **simplified topology** plots in Jupyter

Straight edges connect surviving graph nodes. This intentionally ignores intermediate street geometry so you can see contraction. No figures are written to disk.

In [ ]:
for year, G in sorted(metric_graphs.items()):
    print(f"{CITY} {year}: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
    plot_topology(G, show_nodes=True)


## 5. Load the saved 2022 optimization graph later

In [ ]:
path_2022 = ROOT / '2022' / f'{CITY.lower()}_lts5_2022_SIMPLE_optimization.pkl'
with path_2022.open('rb') as f:
    G_opt_2022 = pickle.load(f)  # Only open your own trusted pickle files
print(type(G_opt_2022), G_opt_2022.number_of_nodes(), G_opt_2022.number_of_edges())
print('Mandatory LTS5:', sum(d['key']==5 and d.get('mandatory_upgrade_to_lts1',False)
                           for _,_,d in G_opt_2022.edges(data=True)))
